In [ ]:
import torch

!nvidia-smi

print("PyTorch version:", torch.__version__)
print("PyTorch CUDA version:", torch.version.cuda)
print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    raise RuntimeError("GPU is unavailable. Do not start training yet.")

In [ ]:
%pip install -q ultralytics==8.4.213 openpyxl pandas pillow pillow-heif pyyaml scikit-learn matplotlib
from google.colab import drive, files
from pathlib import Path
from collections import defaultdict
from datetime import datetime
from PIL import Image
from pillow_heif import register_heif_opener
from ultralytics import YOLO
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import classification_report, confusion_matrix
from IPython.display import display
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import random, re, math, json, yaml, hashlib, shutil, tempfile, gc

register_heif_opener()
drive.mount('/content/drive')
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
DEVICE = 0 if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE)

## Shared setup: paths and experiment settings

In [ ]:
BASE = Path('/content/drive/MyDrive/lecheros/Dataset')
PHOTOS = BASE / 'clean'
LABELS = BASE / 'recortes_v2/yolo_labels'
EXCEL = None
EXPERIMENT = 'YOLO_bcs'
ROOT = BASE / EXPERIMENT
ROOT.mkdir(parents=True, exist_ok=True)
REPORTS = ROOT / 'test_reports'
REPORTS.mkdir(exist_ok=True)
TEST_FRACTION = 0.20
VAL_WITHIN_DEVELOPMENT = 0.20
EPOCHS = 60
PATIENCE = 15
BATCH = 8
DET_SIZE = 640
CLS_SIZE = 224
CONFIDENCE = 0.25
DETECTOR_PRETRAINED = 'yolo26s.pt'
CLASSIFIER_PRETRAINED = 'yolo26s-cls.pt'
DROPOUT = 0.30

if EXCEL is None:
    uploaded = files.upload()
    names = [name for name in uploaded if name.lower().endswith('.xlsx')]
    if len(names) != 1:
        raise ValueError('Upload exactly one Excel file.')
    EXCEL = Path('/content') / names[0]
for path in [PHOTOS, LABELS, EXCEL]:
    if not path.exists():
        raise FileNotFoundError(path)
WORK = Path(tempfile.mkdtemp(prefix='bcs_holdout_'))
DET = WORK / 'detection'
CLS = WORK / 'classification'
print('Persistent experiment:', ROOT)
print('Local prepared data:', WORK)

In [ ]:
def canonical_id(value):
    value = str(value).strip().lower()
    return value[:-2] if value.endswith('.0') else value

def read_box(path):
    rows = [line.split() for line in Path(path).read_text().splitlines() if line.strip()]
    if len(rows) != 1 or len(rows[0]) != 5:
        raise ValueError(f'Expected one class/xc/yc/w/h box: {path}')
    values = np.asarray(rows[0], dtype=float)
    _, xc, yc, bw, bh = values
    if not np.isfinite(values).all() or not (0 <= xc <= 1 and 0 <= yc <= 1 and 0 < bw <= 1 and 0 < bh <= 1):
        raise ValueError(f'Invalid box: {path}')
    if xc-bw/2 < -1e-5 or yc-bh/2 < -1e-5 or xc+bw/2 > 1+1e-5 or yc+bh/2 > 1+1e-5:
        raise ValueError(f'Box outside image: {path}')
    return xc, yc, bw, bh

def pixel_box(box, size):
    xc, yc, bw, bh = box
    w, h = size
    return (max(0, math.floor((xc-bw/2)*w)), max(0, math.floor((yc-bh/2)*h)),
            min(w, math.ceil((xc+bw/2)*w)), min(h, math.ceil((yc+bh/2)*h)))

def square_pad(image):
    side = max(image.size)
    canvas = Image.new('RGB', (side, side), (114, 114, 114))
    canvas.paste(image, ((side-image.width)//2, (side-image.height)//2))
    return canvas

def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, 'rb') as file:
        for block in iter(lambda: file.read(1024*1024), b''):
            digest.update(block)
    return digest.hexdigest()

table = pd.read_excel(EXCEL, sheet_name='Hoja 1')
table.columns = table.columns.str.strip()
table['score'] = pd.to_numeric(table['BCS'], errors='coerce')
table = table.dropna(subset=['NO Vac', 'score']).copy()
source_flag = pd.to_numeric(table['Rancho'], errors='raise')
ranch = table.loc[source_flag.eq(1)].copy()
ranch['cow_id'] = ranch['NO Vac'].map(canonical_id)
if not ranch['cow_id'].is_unique:
    raise ValueError('Duplicate observation IDs in Excel.')
ranch_lookup = ranch.set_index('cow_id')['score'].to_dict()
external_lookup = {}
for _, row in table.loc[source_flag.eq(0)].iterrows():
    name = 'ext_' + str(row['Vacas']).strip().lower()
    if name in external_lookup:
        raise ValueError(f'Duplicate external group: {name}')
    external_lookup[name] = float(row['score'])
print('External BCS mapping:', external_lookup)

label_index = defaultdict(list)
for path in sorted(LABELS.rglob('*')):
    if path.is_file() and path.suffix.lower() == '.txt':
        label_index[path.stem.lower()].append(path)
records, skipped, errors = [], [], []
for source in ['rancho', 'externo']:
    root = PHOTOS / source
    if not root.is_dir():
        raise FileNotFoundError(root)
    for path in sorted(root.rglob('*')):
        if not path.is_file() or path.suffix.lower() not in {'.jpg', '.jpeg', '.png', '.heic'}:
            continue
        try:
            cow_id = canonical_id(path.relative_to(root).parts[0])
            if source == 'rancho':
                match = re.match(r'^(\d+(?:_r\d+)?)(?=_|$)', path.stem, re.I)
                if match is None or canonical_id(match.group(1)) != cow_id:
                    raise ValueError('Folder ID and filename ID disagree.')
                if cow_id not in ranch_lookup:
                    raise ValueError(f'Cow absent from Excel: {cow_id}')
                score = float(ranch_lookup[cow_id])
                animal_id = re.sub(r'_r\d+$', '', cow_id)
            else:
                if cow_id not in external_lookup or not path.stem.lower().startswith(cow_id + '_'):
                    raise ValueError('External folder/filename/Excel mapping disagrees.')
                score = external_lookup[cow_id]
                animal_id = cow_id
            matches = label_index[path.stem.lower()]
            if not matches:
                skipped.append({'image': str(path), 'reason': 'Missing matching TXT annotation'})
                continue
            if len(matches) != 1:
                raise ValueError('Multiple matching TXT files.')
            read_box(matches[0])
            with Image.open(path) as im:
                rgb = im.convert('RGB')
                # Pixel hash detects exact duplicates even across different file encodings.
                pixel_hash = hashlib.sha256(str(rgb.size).encode() + rgb.tobytes()).hexdigest()
            records.append(dict(image=str(path), label=str(matches[0]), source=source,
                                cow_id=cow_id, animal_id=animal_id, bcs=score,
                                image_hash=sha256_file(path), pixel_hash=pixel_hash,
                                label_hash=sha256_file(matches[0])))
        except Exception as exc:
            errors.append({'image': str(path), 'problem': str(exc)})
pd.DataFrame(skipped, columns=['image', 'reason']).to_csv(ROOT / 'skipped_images.csv', index=False)
if errors:
    display(pd.DataFrame(errors))
    raise ValueError('Fix the listed data problems before training.')
manifest = pd.DataFrame(records).sort_values('image').reset_index(drop=True)
if manifest.empty:
    raise ValueError('No eligible labeled images.')
if manifest['image'].map(lambda p: Path(p).stem.lower()).duplicated().any():
    raise ValueError('Duplicate image stems: label associations are ambiguous.')
print('Eligible photos:', len(manifest), '| Missing-label photos excluded:', len(skipped))
missing_cows = sorted(set(ranch_lookup) - set(manifest.loc[manifest.source.eq('rancho'), 'cow_id']))
print('Excel observation IDs without eligible photos:', missing_cows)
if skipped:
    display(pd.DataFrame(skipped).head(10))

## Grouped split and class coverage
Select a label-balanced grouped split from deterministic candidates. All classes must remain represented in fit training (including the training-only external images). Candidate selection uses class counts only. It cannot create missing animals in rare classes; unsupported test classes are reported as such. Do not change the seed after seeing model results.

In [ ]:
def grouped_holdout(frame, fraction, seed, extra_training_scores=()):
    frame = frame.reset_index(drop=True)
    classes = sorted(frame['bcs'].unique())
    overall = frame['bcs'].value_counts(normalize=True).reindex(classes, fill_value=0)
    best = None
    splitter = GroupShuffleSplit(n_splits=2000, test_size=fraction, random_state=seed)
    for fit_idx, held_idx in splitter.split(frame, groups=frame['animal_id']):
        fit, held = frame.iloc[fit_idx], frame.iloc[held_idx]
        if not set(classes).issubset(set(fit['bcs']) | set(extra_training_scores)):
            continue
        distribution = held['bcs'].value_counts(normalize=True).reindex(classes, fill_value=0)
        cost = float(np.abs(distribution-overall).sum() + abs(len(held)/len(frame)-fraction))
        if best is None or cost < best[0]:
            best = (cost, fit.copy(), held.copy())
    if best is None:
        raise ValueError('No feasible grouped split with training class coverage. Inspect animal counts; do not split photos of one cow.')
    return best[1], best[2]

ranch_data = manifest.loc[manifest.source.eq('rancho')].copy()
external_data = manifest.loc[manifest.source.eq('externo')].copy()
external_scores = set(external_data['bcs'])
development, test_data = grouped_holdout(ranch_data, TEST_FRACTION, SEED, external_scores)
fit_data, val_data = grouped_holdout(development, VAL_WITHIN_DEVELOPMENT, SEED, external_scores)
assignments = {}
for split, frame in [('train', fit_data), ('val', val_data), ('test', test_data)]:
    for animal in frame['animal_id'].unique():
        assignments[animal] = split
manifest['split'] = [assignments[row.animal_id] if row.source == 'rancho' else 'train'
                     for row in manifest.itertuples()]
assert manifest.loc[manifest.source.eq('rancho')].groupby('animal_id')['split'].nunique().max() == 1
assert set(manifest['split']) == {'train', 'val', 'test'}
assert set(manifest['bcs']).issubset(set(manifest.loc[manifest.split.eq('train'), 'bcs']))
# Stop if exactly identical pixels cross splits, including external versus ranch.
cross_duplicates = manifest.groupby('pixel_hash')['split'].nunique()
if cross_duplicates.gt(1).any():
    display(manifest.loc[manifest.pixel_hash.isin(cross_duplicates[cross_duplicates.gt(1)].index)])
    raise ValueError('Exact duplicate images cross splits. Resolve duplicate identities before training.')

config = dict(seed=SEED, test_fraction=TEST_FRACTION, validation_fraction=VAL_WITHIN_DEVELOPMENT,
              epochs=EPOCHS, patience=PATIENCE, batch=BATCH, detector=DETECTOR_PRETRAINED,
              classifier=CLASSIFIER_PRETRAINED, dropout=DROPOUT, detector_size=DET_SIZE,
              classifier_size=CLS_SIZE, confidence=CONFIDENCE, ultralytics='8.4.213',
              external_policy='train_only', split_candidates=2000, implementation='holdout_v1')
manifest_text = manifest.to_csv(index=False)
fingerprint = hashlib.sha256((json.dumps(config, sort_keys=True)+manifest_text).encode()).hexdigest()
config_path = ROOT / 'experiment.json'
if config_path.exists():
    if json.loads(config_path.read_text())['fingerprint'] != fingerprint:
        raise ValueError('Data/settings differ from this saved experiment. Use a new EXPERIMENT name.')
else:
    config_path.write_text(json.dumps(dict(config=config, fingerprint=fingerprint), indent=2))
manifest.to_csv(ROOT / 'manifest.csv', index=False)
shutil.copy2(EXCEL, ROOT / 'expert_scores.xlsx')
print('Photo counts:')
display(pd.crosstab([manifest.source, manifest.split], manifest.bcs))
group_counts = manifest.loc[manifest.source.eq('rancho')].groupby('split')['animal_id'].nunique()
print('Ranch physical cows by split:')
display(group_counts.to_frame('Cows').assign(Percent=100*group_counts/group_counts.sum()))
coverage = pd.crosstab(manifest.bcs, manifest.split).reindex(columns=['train','val','test'], fill_value=0)
coverage['test_status'] = np.where(coverage['test'].eq(0), 'NOT EVALUATED: no test photos', 'Evaluated')
display(coverage)
coverage.to_csv(ROOT / 'class_split_counts.csv')

In [ ]:
scores = sorted(manifest['bcs'].unique())
class_names = [f'{v:.2f}' for v in scores]
for split in ['train', 'val', 'test']:
    (DET / 'images' / split).mkdir(parents=True, exist_ok=True)
    (DET / 'labels' / split).mkdir(parents=True, exist_ok=True)
    for name in class_names:
        (CLS / split / name).mkdir(parents=True, exist_ok=True)
prepared = []
for i, row in manifest.iterrows():
    box = read_box(row.label)
    with Image.open(row.image) as original:
        image = original.convert('RGB')  # Match original annotation coordinates; no EXIF transpose.
    stem = f'image_{i:06d}'
    detection_image = DET / 'images' / row.split / f'{stem}.jpg'
    manual_crop = CLS / row.split / f'{row.bcs:.2f}' / f'{stem}.jpg'
    image.save(detection_image)
    (DET / 'labels' / row.split / f'{stem}.txt').write_text('0 ' + ' '.join(f'{x:.8f}' for x in box) + '\n')
    square_pad(image.crop(pixel_box(box, image.size))).save(manual_crop)
    prepared.append(dict(detection_image=str(detection_image), manual_crop=str(manual_crop)))
prepared = pd.concat([manifest.reset_index(drop=True), pd.DataFrame(prepared)], axis=1)
(DET / 'data.yaml').write_text(yaml.safe_dump(dict(path=str(DET), train='images/train',
                                                val='images/val', test='images/test', names={0:'cow'})))
# Preview only training crops before model fitting.
preview = prepared.loc[prepared.split.eq('train')].sample(min(4, (prepared.split=='train').sum()), random_state=SEED)
fig, axes = plt.subplots(1, len(preview), figsize=(12, 4), squeeze=False)
for ax, row in zip(axes[0], preview.itertuples()):
    with Image.open(row.manual_crop) as im:
        ax.imshow(im)
    ax.set_title(f'{row.cow_id}: BCS {row.bcs:.2f}')
    ax.axis('off')
plt.tight_layout()
plt.show()
print('Prepared training, validation and test files.')

## Training helper
Only a completed checkpoint from this exact new experiment may be reused after reconnection. Old fold weights are never loaded. Partial runs are preserved in a backup folder and restarted from public pretrained weights. Training uses validation only; test evaluation is at the end.

In [ ]:
def train_model(name, pretrained, data, imgsz, dropout=None):
    run = ROOT / name
    marker = run / 'completed.json'
    best = run / 'weights/best.pt'
    settings = dict(name=name, pretrained=pretrained, imgsz=imgsz, dropout=dropout,
                    fingerprint=fingerprint)
    if marker.is_file() and best.is_file():
        if json.loads(marker.read_text()) != settings:
            raise ValueError(f'Checkpoint settings mismatch: {run}')
        print('Reusing completed model from this experiment:', best)
        return best
    if run.exists():
        backup = run.with_name(f'{name}_partial_{datetime.now():%Y%m%d_%H%M%S_%f}')
        run.rename(backup)
        print('Preserved incomplete run:', backup)
    model = YOLO(pretrained)
    kwargs = dict(data=str(data), epochs=EPOCHS, patience=PATIENCE, imgsz=imgsz,
                  batch=BATCH, device=DEVICE, workers=2, seed=SEED,
                  deterministic=True, project=str(ROOT), name=name, plots=True)
    if dropout is not None:
        kwargs['dropout'] = dropout
    model.train(**kwargs)
    actual_best = Path(model.trainer.best)
    if actual_best.resolve() != best.resolve() or not best.is_file():
        raise RuntimeError(f'Unexpected checkpoint path: {actual_best}')
    if dropout is not None:
        (run / 'bcs_class_mapping.json').write_text(json.dumps(model.names, indent=2))
    shutil.copy2(ROOT / 'manifest.csv', run / 'manifest.csv')
    marker.write_text(json.dumps(settings, indent=2))
    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return best

# Part 3 — Dropout BCS classifier
Same prepared data and settings as the baseline, with dropout 0.30. Start independently from the same public pretrained classifier.

In [ ]:
dropout_path = train_model('classifier_dropout', CLASSIFIER_PRETRAINED, CLS, CLS_SIZE, dropout=DROPOUT)
print(dropout_path)

# Final test evaluation
Run after all training is complete. Do not use test results to select epochs or adjust thresholds. The detector chooses the highest-confidence cow box at confidence ≥0.25; it never chooses using the expert box. In scenes with multiple cows, this rule can choose a different animal; IoU quantifies alignment with the annotated target.

Detection metrics: mAP50, mAP50–95, precision and recall from YOLO validation, plus fixed-threshold selected-box coverage and IoU. The standard detector confusion matrix is saved in `detector_test`.

BCS metrics: MAE, RMSE, tolerated accuracy ±0.25, exact accuracy, coverage, macro/weighted F1, per-class precision/recall/F1/support, and count/row-normalized confusion matrices. Classifiers are tested both on manual crops and automatic detector crops. Missing detections count as incorrect in accuracy and F1; MAE/RMSE use available predictions and are explicitly conditional on coverage.

In [ ]:
# Saved models from the detector and baseline notebooks.
detector_path = ROOT / "detector" / "weights" / "best.pt"
baseline_path = ROOT / "classifier_baseline" / "weights" / "best.pt"

for name, path in {
    "Detector": detector_path,
    "Baseline classifier": baseline_path,
}.items():
    if not path.is_file():
        raise FileNotFoundError(
            f"{name} not found:\n{path}\n"
            "Check that all three notebooks use the same EXPERIMENT folder."
        )

    # Verify the model belongs to the same data split and settings.
    marker = path.parent.parent / "completed.json"
    if not marker.is_file():
        raise FileNotFoundError(f"Missing training completion record: {marker}")

    if json.loads(marker.read_text()).get("fingerprint") != fingerprint:
        raise ValueError(f"{name} was trained with different data or settings.")

    print(f"{name}: {path}")

In [ ]:
test = prepared.loc[prepared.split.eq('test')].copy().reset_index(drop=True)
detector = YOLO(str(detector_path))
det_metrics = detector.val(data=str(DET / 'data.yaml'), split='test', imgsz=DET_SIZE,
                           batch=BATCH, device=DEVICE, workers=2, plots=True,
                           project=str(ROOT), name='detector_test', exist_ok=True)
det_summary = dict(mAP50=float(det_metrics.box.map50), mAP50_95=float(det_metrics.box.map),
                   precision=float(det_metrics.box.mp), recall=float(det_metrics.box.mr))

def box_iou(a, b):
    x1, y1 = max(a[0],b[0]), max(a[1],b[1])
    x2, y2 = min(a[2],b[2]), min(a[3],b[3])
    intersection = max(0,x2-x1)*max(0,y2-y1)
    area_a = max(0,a[2]-a[0])*max(0,a[3]-a[1])
    area_b = max(0,b[2]-b[0])*max(0,b[3]-b[1])
    return intersection / (area_a+area_b-intersection) if area_a+area_b-intersection > 0 else 0.0

auto_dir = WORK / 'automatic_test_crops'
auto_dir.mkdir(exist_ok=True)
auto_paths, detection_rows = [], []
for i, row in test.iterrows():
    result = detector.predict(row.detection_image, imgsz=DET_SIZE, conf=CONFIDENCE,
                              device=DEVICE, verbose=False)[0]
    boxes = result.boxes
    count = 0 if boxes is None else len(boxes)
    predicted_path, selected_iou, confidence = None, np.nan, np.nan
    if count:
        j = int(boxes.conf.argmax().item())
        xyxy = boxes.xyxy[j].cpu().numpy().tolist()
        confidence = float(boxes.conf[j].item())
        with Image.open(row.detection_image) as im:
            image = im.convert('RGB')
        w,h = image.size
        crop_box = (max(0,math.floor(xyxy[0])), max(0,math.floor(xyxy[1])),
                    min(w,math.ceil(xyxy[2])), min(h,math.ceil(xyxy[3])))
        if crop_box[2] > crop_box[0] and crop_box[3] > crop_box[1]:
            predicted_path = str(auto_dir / f'{i:06d}.jpg')
            square_pad(image.crop(crop_box)).save(predicted_path)
            selected_iou = box_iou(xyxy, pixel_box(read_box(row.label), image.size))
    auto_paths.append(predicted_path)
    detection_rows.append(dict(image=row.image, detected=predicted_path is not None,
                               boxes=count, confidence=confidence, iou=selected_iou))
detection_results = pd.DataFrame(detection_rows)
det_summary.update(test_photos=len(test), coverage_pct=100*detection_results.detected.mean(),
                   selected_box_mean_iou=detection_results.iou.mean(),
                   iou_ge_050_all_photos_pct=100*detection_results.iou.fillna(0).ge(.5).mean())
display(pd.DataFrame([det_summary]))
pd.DataFrame([det_summary]).to_csv(REPORTS / 'detector_metrics.csv', index=False)
detection_results.to_csv(REPORTS / 'detector_predictions.csv', index=False)
del detector
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

In [ ]:
def predict_bcs(model, paths):
    predictions = np.full(len(paths), np.nan)
    valid = [(i,p) for i,p in enumerate(paths) if p is not None]
    for start in range(0, len(valid), BATCH):
        items = valid[start:start+BATCH]
        results = model.predict([p for _,p in items], imgsz=CLS_SIZE, device=DEVICE,
                                augment=False, verbose=False)
        if len(results) != len(items):
            raise ValueError('Incomplete classifier outputs.')
        for (i,_), result in zip(items, results):
            if result.probs is None:
                raise ValueError('Expected classification probabilities.')
            predictions[i] = float(result.names[result.probs.top1])
    return predictions

predictions = test[['image','cow_id','animal_id','bcs']].copy()
for name, path in [('Baseline',baseline_path), ('Dropout',dropout_path)]:
    classifier = YOLO(str(path))
    if classifier.task != 'classify':
        raise ValueError('Expected a classifier.')
    if set(float(v) for v in classifier.names.values()) != set(scores):
        raise ValueError(f'{name}: trained class set differs from prepared class set.')
    predictions[f'{name}_manual'] = predict_bcs(classifier, test.manual_crop.tolist())
    predictions[f'{name}_automatic'] = predict_bcs(classifier, auto_paths)
    del classifier
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
predictions.to_csv(REPORTS / 'bcs_predictions.csv', index=False)
print('Saved test predictions for both classifiers and both crop methods.')

In [ ]:
prediction_columns = ['Baseline_manual','Dropout_manual','Baseline_automatic','Dropout_automatic']
def bcs_metrics(frame, column):
    y = frame['bcs'].to_numpy(dtype=float)
    pred = frame[column].to_numpy(dtype=float)
    valid = np.isfinite(pred)
    errors = pred[valid]-y[valid]
    tolerance_hits = int(np.sum(np.abs(errors) <= .25+1e-9))
    exact_hits = int(np.sum(np.isclose(errors,0,atol=1e-9,rtol=0)))
    n = len(frame)
    return dict(Photos=n, Predictions=int(valid.sum()), Coverage_pct=100*valid.mean() if n else np.nan,
                MAE=float(np.abs(errors).mean()) if valid.any() else np.nan,
                RMSE=float(np.sqrt(np.mean(errors**2))) if valid.any() else np.nan,
                Within_025_pct=100*tolerance_hits/n if n else np.nan,
                Exact_accuracy_pct=100*exact_hits/n if n else np.nan,
                Within_025_predicted_only_pct=100*tolerance_hits/valid.sum() if valid.any() else np.nan)

summary_rows=[]
for scope, frame in [('All BCS',predictions), ('BCS 3.25–3.50',predictions.loc[predictions.bcs.between(3.25,3.50)])]:
    for column in prediction_columns:
        row = dict(Scope=scope, Model=column, **bcs_metrics(frame,column))
        row['Criteria_status'] = 'Not applicable'
        if scope == 'BCS 3.25–3.50':
            if row['Photos'] == 0:
                row['Criteria_status'] = 'NOT EVALUATED'
            elif row['Predictions'] < row['Photos']:
                row['Criteria_status'] = 'INCOMPLETE: conditional MAE/RMSE'
            else:
                row['Criteria_status'] = 'PASS' if row['MAE'] <= .25+1e-9 and row['RMSE'] <= .35+1e-9 and row['Within_025_pct'] >= 80-1e-9 else 'FAIL'
        summary_rows.append(row)
summary = pd.DataFrame(summary_rows)
display(summary.round(4))
summary.to_csv(REPORTS / 'bcs_metrics.csv', index=False)

# Direct paired changes: Dropout minus Baseline, on the same test images.
changes=[]
for scope in summary.Scope.unique():
    for crop in ['manual','automatic']:
        a = summary.loc[(summary.Scope==scope)&(summary.Model==f'Baseline_{crop}')].iloc[0]
        b = summary.loc[(summary.Scope==scope)&(summary.Model==f'Dropout_{crop}')].iloc[0]
        for metric in ['MAE','RMSE','Within_025_pct','Exact_accuracy_pct']:
            delta=b[metric]-a[metric]
            direction='Not evaluated' if pd.isna(delta) else ('Unchanged' if abs(delta)<1e-9 else ('Improved' if (delta<0 if metric in ['MAE','RMSE'] else delta>0) else 'Worse'))
            changes.append(dict(Scope=scope,Crop=crop,Metric=metric,Dropout_minus_Baseline=delta,Result=direction))
changes=pd.DataFrame(changes)
display(changes.round(4))
changes.to_csv(REPORTS / 'baseline_vs_dropout.csv',index=False)

# Numeric criteria individually, only for the target expert interval.
criteria=summary.loc[summary.Scope.eq('BCS 3.25–3.50'), ['Model','Photos','Predictions','MAE','RMSE','Within_025_pct','Criteria_status']].copy()
criteria['MAE_le_025']=criteria.MAE.le(.25+1e-9)
criteria['RMSE_le_035']=criteria.RMSE.le(.35+1e-9)
criteria['Tolerated_ge_80pct']=criteria.Within_025_pct.ge(80-1e-9)
display(criteria.round(4))
criteria.to_csv(REPORTS / 'success_criteria.csv',index=False)

## Confusion matrices and per-class F1
Rows = expert BCS; columns = predicted BCS. Automatic-crop matrices include a “No prediction” column. Each row label includes F1. Per-class tables include precision, recall, F1, support and evaluation status. Zero-support classes are not evaluated; sklearn's zero-filled F1 values for those rows are not evidence of poor performance. Macro F1 below averages only classes actually present in the test set; weighted F1 uses support.

In [ ]:
def classification_details(frame, column):
    truth = [f'{v:.2f}' for v in frame.bcs]
    predicted = [f'{v:.2f}' if np.isfinite(v) else 'No prediction' for v in frame[column]]
    report = classification_report(truth, predicted, labels=class_names, output_dict=True, zero_division=0)
    per_class = pd.DataFrame([dict(BCS=name, **report[name]) for name in class_names])
    per_class['support'] = per_class['support'].astype(int)
    per_class['Status'] = np.where(per_class.support.gt(0), 'Evaluated', 'No test photos')
    supported=per_class.loc[per_class.support.gt(0)]
    macro=float(supported['f1-score'].mean()) if len(supported) else np.nan
    weighted=float(np.average(supported['f1-score'],weights=supported.support)) if len(supported) else np.nan
    print(column, '| macro F1 (supported classes):', round(macro,4), '| weighted F1:',round(weighted,4))
    display(per_class.round(4))
    per_class.to_csv(REPORTS / f'{column}_per_class.csv',index=False)
    labels=class_names+(['No prediction'] if 'No prediction' in predicted else [])
    matrix=confusion_matrix(truth,predicted,labels=labels)[:len(class_names),:]
    normalized=np.divide(matrix,matrix.sum(axis=1,keepdims=True),out=np.zeros_like(matrix,dtype=float),where=matrix.sum(axis=1,keepdims=True)>0)
    fig,axes=plt.subplots(1,2,figsize=(19,8))
    row_labels=[f"{r['BCS']} | F1={r['f1-score']:.2f}" if r['support']>0 else f"{r['BCS']} | F1=N/A" for _,r in per_class.iterrows()]
    for ax,data,title in zip(axes,[matrix,normalized],['Counts','Row proportions']):
        im=ax.imshow(data,cmap='Blues',vmin=0,vmax=1 if title=='Row proportions' else None)
        ax.set_xticks(range(len(labels)),labels,rotation=45,ha='right')
        ax.set_yticks(range(len(class_names)),row_labels)
        ax.set_xlabel('Predicted BCS'); ax.set_ylabel('Expert BCS')
        ax.set_title(f'{column} — {title}')
        for i in range(data.shape[0]):
            for j in range(data.shape[1]):
                value=f'{data[i,j]:.0%}' if title=='Row proportions' else str(int(data[i,j]))
                ax.text(j,i,value,ha='center',va='center',color='white' if data[i,j]>data.max()/2 else 'black',fontsize=8)
        fig.colorbar(im,ax=ax,fraction=.03)
    plt.tight_layout()
    fig.savefig(REPORTS / f'{column}_confusion.png',dpi=160,bbox_inches='tight')
    plt.show()
    pd.DataFrame(matrix,index=class_names,columns=labels).to_csv(REPORTS / f'{column}_confusion.csv')
    return dict(Model=column,Macro_F1_supported=macro,Weighted_F1=weighted)

f1_summary=pd.DataFrame([classification_details(predictions,column) for column in prediction_columns])
f1_summary.to_csv(REPORTS / 'f1_summary.csv',index=False)
display(f1_summary.round(4))
print('All outputs saved in:', ROOT)
print('Test metrics, per-class reports, predictions and confusion matrices:', REPORTS)